# 01 - Feature Engineering

Feature engineering turns the raw competition files into columns that models can learn from. In this project it should happen after EDA and before serious model training.

The most important rule is no target leakage. Any feature based on `Weekly_Sales` must use only historical sales that would already be known at prediction time.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dataloader import load_raw, merge_all
from features import build_features, HOLIDAY_WEEK_BY_DATE, MARKDOWN_COLS
from metrics import wmae
from validation import holdout_split

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 120)

In [ ]:
train, test, feature_table, stores, sample_submission = load_raw(PROJECT_ROOT / 'data')
train_merged = merge_all(train, feature_table, stores)
test_merged = merge_all(test, feature_table, stores)

print(train_merged.shape, test_merged.shape)
train_merged.head()

## What feature groups we create and why

1. Calendar features: retail sales have yearly and monthly seasonality.
2. Holiday flags: the four flagged holidays have different effects and WMAE weights them more heavily.
3. Store metadata: store type and size explain baseline sales level.
4. External variables: temperature, fuel price, CPI, unemployment, and markdowns may explain demand changes.
5. Derived weather/fuel features: extreme cold/hot flags, comfort-zone distance, store-level deviations, and interaction terms help models learn nonlinear effects from `Temperature` and `Fuel_Price`.
6. Missingness indicators: missing markdown/economic data is informative, not just an error.
7. Sales history: lag 52 and historical averages are strong because the same store-department often behaves similarly one year later.

In [ ]:
X_train = build_features(train_merged, encode_categoricals=True)
X_test = build_features(test_merged, sales_history_df=train_merged, encode_categoricals=True)

print('train matrix:', X_train.shape)
print('test matrix:', X_test.shape)
print('same columns:', list(X_train.columns) == list(X_test.columns))
print('missing train cells:', int(X_train.isna().sum().sum()))
print('missing test cells:', int(X_test.isna().sum().sum()))
X_train.head()

In [ ]:
groups = {
    'id/store': ['Store', 'Dept', 'Size'] + [c for c in X_train.columns if c.startswith('Type_')],
    'calendar': ['Year', 'Month', 'Quarter', 'WeekOfYear', 'DayOfYear', 'DaysFromStart', 'WeekSin', 'WeekCos', 'MonthSin', 'MonthCos'],
    'holiday': [c for c in X_train.columns if c.startswith('is_') or c.startswith('HolidayName_') or c == 'IsHoliday'],
    'external': ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment'],
    'markdown': [c for c in X_train.columns if 'MarkDown' in c or 'Markdown' in c],
    'sales_history': [c for c in X_train.columns if 'sales_lag' in c or 'history_mean' in c],
}

pd.DataFrame([(group, col) for group, cols in groups.items() for col in cols if col in X_train.columns], columns=['group', 'feature'])

## Your average-week idea, implemented safely

A useful idea is: "for this Store-Dept and week of year, what was the average sale in earlier years?"

Unsafe version: average all years in the full training set. That leaks future data into earlier rows.

Safe version used here: `same_week_history_mean`, an expanding mean by `(Store, Dept, WeekOfYear)` that only uses dates before the current row. For test rows it uses train history only.

In [ ]:
example = train_merged[(train_merged['Store'] == 1) & (train_merged['Dept'] == 1)].copy()
example_X = build_features(example, encode_categoricals=True)
example = example.join(example_X[['sales_lag_52', 'same_week_history_mean', 'store_dept_history_mean']])

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(example['Date'], example['Weekly_Sales'], label='actual', marker='o', linewidth=1)
ax.plot(example['Date'], example['sales_lag_52'], label='lag 52', linewidth=1)
ax.plot(example['Date'], example['same_week_history_mean'], label='same-week historical mean', linewidth=1)
ax.set_title('Example safe history features for Store 1 / Dept 1')
ax.legend()
plt.show()

## Holiday flags

`IsHoliday` tells us whether the week is one of the competition holidays. Separate holiday columns tell the model which holiday it is. Thanksgiving and Christmas behave very differently, so one generic holiday flag is too weak.

In [ ]:
holiday_check = train_merged[['Date', 'IsHoliday']].drop_duplicates().sort_values('Date')
holiday_check['HolidayName'] = holiday_check['Date'].dt.strftime('%Y-%m-%d').map(HOLIDAY_WEEK_BY_DATE).fillna('none')
holiday_check[holiday_check['IsHoliday']]

## Derived temperature and fuel-price features

Raw `Temperature` and `Fuel_Price` are included, but we also create features that are easier to explain:

- `TempCold`, `TempHot`, `TempMild`: simple weather regimes.
- `TempComfortDistance`: how far the week is from a comfortable 55-75?F range.
- `TemperatureStoreDeviation`: whether this week is hotter/colder than this store usually is.
- `FuelPriceStoreDeviation`: whether fuel is more/less expensive than usual for this store.
- `TempFuelInteraction` and `ComfortDistanceFuelInteraction`: allow the model to learn that weather and fuel cost can matter together.

These are safe because they use external variables available in `features.csv` for both train and test. They still need feature selection later; we include them first, then validate whether they help WMAE.

In [ ]:
weather_fuel_cols = [
    'Temperature', 'Fuel_Price', 'TempCold', 'TempHot', 'TempMild',
    'TempComfortDistance', 'TemperatureStoreDeviation',
    'FuelPriceStoreDeviation', 'FuelPriceHigh',
    'TempFuelInteraction', 'ComfortDistanceFuelInteraction',
]
X_train[[c for c in weather_fuel_cols if c in X_train.columns]].describe().T

## Markdowns and economic variables

Markdown values are filled with 0, and missing flags are added. CPI and unemployment are forward/backward filled per store because some future values are missing in `features.csv`. The missing flags remain, so the model can learn that the value was imputed.

In [ ]:
missing_features = [c for c in X_train.columns if c.endswith('_missing')]
X_train[missing_features].mean().sort_values(ascending=False).head(20).to_frame('share_missing_or_fallback')

## Quick leakage sanity checks

Early 2010 rows should have fallback values for lag 52 because there is no previous year in the dataset. Later rows should have real lag 52 values more often. This is expected and is exactly why we keep missing/fallback flags.

In [ ]:
lag_check = train_merged[['Store', 'Dept', 'Date', 'Weekly_Sales']].join(X_train[['sales_lag_52', 'sales_lag_52_missing', 'same_week_history_mean_missing']])
lag_check.groupby(train_merged['Date'].dt.year)[['sales_lag_52_missing', 'same_week_history_mean_missing']].mean()

## Baseline validation using the engineered features

Before fitting complex models, score a simple feature-based baseline. Here we predict with lag 52 and use WMAE on the last 39 weeks, which mirrors the real test horizon.

In [ ]:
train_mask, val_mask = holdout_split(train_merged['Date'], horizon=39)
val_pred = X_train.loc[val_mask, 'sales_lag_52'].to_numpy()
val_pred = np.clip(val_pred, 0, None)
val_score = wmae(
    train_merged.loc[val_mask, 'Weekly_Sales'],
    val_pred,
    train_merged.loc[val_mask, 'IsHoliday'],
)
print(f'Lag-52 holdout WMAE: {val_score:,.2f}')

## Optional artifact saving

Most model notebooks can import `build_features` directly instead of saving feature files. Saving parquet files can still be useful for debugging. Keep generated data out of git because the `data/` folder is ignored.

In [ ]:
SAVE_FEATURES = False

if SAVE_FEATURES:
    processed_dir = PROJECT_ROOT / 'data' / 'processed'
    processed_dir.mkdir(parents=True, exist_ok=True)
    X_train.assign(Weekly_Sales=train_merged['Weekly_Sales'].values).to_parquet(processed_dir / 'train_features.parquet', index=False)
    X_test.to_parquet(processed_dir / 'test_features.parquet', index=False)
    print(processed_dir)